In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.monitorizacion.evaluacion import calcular_metricas

%load_ext autoreload
%autoreload 2

In [ ]:
ROOT = Path.cwd().parent
sim = pd.read_parquet(ROOT/"reports"/"sim_semanal.parquet")
sim = sim[["datetime_utc", "fecha_objetivo", "prediccion", "naive_d1", "precio_real"]]

In [ ]:
print(sim.shape)
print(sim.head())
print(sim["fecha_objetivo"].min())
print(sim["fecha_objetivo"].max())

In [ ]:
metricas = calcular_metricas(sim)
print(len(metricas))
metricas[["regret_compra_k4", "regret_venta_k4", "regret_compra_naive_k4", "regret_venta_naive_k4"]].head()

In [ ]:
MWH_DIA = 4

metricas["perdida_modelo"] = (metricas["regret_compra_k4"] + metricas["regret_venta_k4"]) * MWH_DIA
metricas["perdida_naive"] = (metricas["regret_compra_naive_k4"] + metricas["regret_venta_naive_k4"]) * MWH_DIA

print(f"la perdida  del modelo son {metricas['perdida_modelo'].mean():.2f}€ diarios de media")
print(f"la perdida de naive son {metricas['perdida_naive'].mean():.2f}€ diarios de media")

In [ ]:
metricas["ahorro_diario"] = metricas["perdida_naive"] - metricas["perdida_modelo"]
ahorro_medio_diario = metricas["ahorro_diario"].mean()
ahorro_anual = ahorro_medio_diario * 365
porcentaje_ahorro = 1 - metricas["perdida_modelo"].mean() / metricas["perdida_naive"].mean() 

print(f"El ahorro medio diario del modelo frente al naive es de {ahorro_medio_diario:.1f}€ ")
print(f"El ahorro anual del modelo frente al naive es de {ahorro_anual:.0f}€ ")
print(f"El porcentaje de ahorro del modelo frente al naive es de {porcentaje_ahorro:.0%} ")


In [ ]:
perdidas = metricas[["perdida_modelo","perdida_naive"]]

riesgo_cola = pd.DataFrame({"mediana": perdidas.median(),
                            "media": perdidas.mean(),
                            "percentil_90": perdidas.quantile(0.90),
                            "percentil_95": perdidas.quantile(0.95),
                            "maximo": perdidas.max()
                            }).T.round(1)
riesgo_cola

In [ ]:
display(riesgo_cola)
print("-----------------------dias con perdidas mayores a 100€-----------------------")
(perdidas >100).sum()

In [ ]:
sensibilidad = pd.DataFrame({"MW": [1, 10, 50]})
sensibilidad["MWh_dia"] = sensibilidad["MW"] * MWH_DIA
sensibilidad["ahorro_anual_eur"] = ahorro_anual * sensibilidad["MW"]
sensibilidad.style.format(precision=0, thousands=".", decimal=",").hide(axis="index")

In [ ]:
produccion = pd.read_parquet(ROOT / "data" / "monitoring" / "evaluacion_diaria.parquet")

In [ ]:
produccion["perdida_modelo"] = (produccion["regret_compra_k4"] + produccion["regret_venta_k4"]) * MWH_DIA
produccion["perdida_naive"] = (produccion["regret_compra_naive_k4"] + produccion["regret_venta_naive_k4"]) * MWH_DIA
print(produccion[["perdida_modelo","perdida_naive"]].head())

In [ ]:
def resumen_bateria(perdidas):
    """Misma receta para simulación y producción: si las cifras difieren, es por los datos, no por el método."""
    return pd.Series({
        "días evaluados": len(perdidas),
        "pérdida media modelo (€/día)": perdidas["perdida_modelo"].mean(),
        "pérdida media naive (€/día)": perdidas["perdida_naive"].mean(),
        "reducción de pérdidas (%)": (1 - perdidas["perdida_modelo"].mean() / perdidas["perdida_naive"].mean()) * 100,
        "días > 100 € modelo": (perdidas["perdida_modelo"] > 100).sum(),
        "días > 100 € naive": (perdidas["perdida_naive"] > 100).sum(),
    })


comparacion = pd.DataFrame({
    "Simulación": resumen_bateria(metricas[["perdida_modelo", "perdida_naive"]]),
    "Producción": resumen_bateria(produccion[["perdida_modelo", "perdida_naive"]]),
}).round(1)
comparacion

In [ ]:
def diferencial_perfecto(precios, k=4):
    """€/MWh que ganaría quien conociera el futuro: vender en las k horas más caras y comprar en las k más baratas."""
    return precios.nlargest(k).mean() - precios.nsmallest(k).mean()


metricas["ingreso_perfecto"] = sim.groupby("fecha_objetivo")["precio_real"].apply(diferencial_perfecto) * MWH_DIA
ingreso_perfecto_total = metricas["ingreso_perfecto"].sum()

# Tasa de captura: % del ingreso perfecto que se consigue. Reutiliza las pérdidas (regret) ya calculadas:
# lo capturado es el ingreso perfecto menos lo que se pierde por no conocer el futuro.
captura_modelo = 1 - metricas["perdida_modelo"].sum() / ingreso_perfecto_total
captura_naive = 1 - metricas["perdida_naive"].sum() / ingreso_perfecto_total

print(f"Ingreso con previsión perfecta: {metricas['ingreso_perfecto'].mean() * 365:,.0f} €/año por MW (bruto)".replace(",", "."))
print(f"Tasa de captura del modelo: {captura_modelo:.1%}")
print(f"Tasa de captura del naive:  {captura_naive:.1%}")

## Supuestos y límites del caso de negocio

**Cliente tipo:** batería de 1 MW / 4 MWh que carga en las 4 horas que prevé más baratas y descarga en las 4 más caras, un ciclo al día. Las cifras escalan linealmente con el tamaño (tabla de sensibilidad) mientras se cumplan los supuestos.

**Supuestos**
- **Agente precio-aceptante:** mover la carga o la descarga no cambia el precio de casación de OMIE. Razonable para baterías pequeñas; para volúmenes que muevan la casación habría que modelar el impacto en el precio.
- **Sin pérdidas de eficiencia** (ida y vuelta) **ni costes de operación o degradación** de la batería: las cifras son de **ingreso bruto**.
- **Un ciclo al día con 4 horas fijas** de carga y de descarga, sin restricciones de estado de carga entre días ni participación en otros mercados (servicios de ajuste, intradiario).

**Datos**
- Estimación principal: **635 días simulados** (1-ene-2025 → 5-oct-2026) con la política de producción (reentreno semanal), entrenando cada día solo con el pasado (*point-in-time*).
- Comprobación: **9 días de producción real** (7 → 17-oct-2026). Confirman la simulación en dirección y orden de magnitud (−38 % frente a −41 %), pero son demasiado pocos para dar una cifra propia: un solo día mueve la media.

**Límites**
- El benchmark es el **naive** (precio de ayer), no un proveedor comercial ni el modelo interno de una empresa.
- Frente al estado del arte (Lago et al., 2021: rMAE 0,38-0,61 frente al naive semanal), el modelo queda lejos (rMAE 0,83 frente al naive diario). El límite es la **información**, no el algoritmo: no usa previsión solar de mañana (descartada por *leakage*), ni precios del gas y del CO₂, ni interconexiones.
- En los **días extremos** el modelo apenas mejora al naive (peor día: 255 € frente a 286 €): su valor está en los días malos **habituales** (percentil 90: 56 € frente a 104 €).
- El MAE no mide el dinero: hay días con peor MAE y menor pérdida, y al revés. Por eso la métrica de negocio es el **regret** / **tasa de captura**.

In [ ]:
resumen = pd.Series({
    "ahorro_medio_diario_eur": ahorro_medio_diario,
    "ahorro_anual_eur_por_mw": ahorro_anual,
    "reduccion_perdida_pct": porcentaje_ahorro,
    "captura_modelo": captura_modelo,
    "captura_naive": captura_naive,
})
resumen

In [ ]:
sensibilidad.to_csv(ROOT / "reports"/ "caso_negocio_sensibilidad.csv",index=False)
comparacion.to_csv(ROOT / "reports" / "caso_negocio_sim_vs_produccion.csv")
riesgo_cola.to_csv(ROOT / "reports" / "caso_negocio_riesgo_cola.csv")
resumen.to_csv(ROOT / "reports" / "caso_negocio_resumen.csv",header=["valor"])